# Reference-free quality protocol: 50 vs 20 vs 10 steps vs step caching, Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_quality_2026-11-13.md` (Q1–Q9).

Time: about 10 min to build SageAttention, then about 1 h 50 min of runs (~4.2 compute units). No cell waits for input, so "Run all" can be left alone. Runs are ordered by importance, in case the session ends early.

Each configuration: 4 prompts × 4 seeds = 16 videos at 33 frames, compile + Sage fp16-PV.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit --format=csv

In [ ]:
!pip install -q diffusers==0.42.1 transformers==5.22.2 accelerate ftfy imageio imageio-ffmpeg lpips 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, 'cuda', torch.version.cuda, diffusers.__version__, transformers.__version__)

In [ ]:
%cd /content
!git -C video-dit-l4 pull || git clone https://github.com/FredWen-S/video-dit-l4.git
%cd /content/video-dit-l4

## Build SageAttention for sm_89 (~10 min)

Same build and commit as before. Stops the notebook if it fails.

In [ ]:
import os, re, subprocess, sys, torch
maj, mi = torch.version.cuda.split('.')[:2]
m = re.search(r'release (\d+)\.(\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
print('torch CUDA', torch.version.cuda, '| nvcc', m.group(0) if m else 'not found')
if not m or m.group(1) != maj:
    ubuntu = subprocess.run('. /etc/os-release && echo ubuntu${VERSION_ID/./}', shell=True, capture_output=True,
                            text=True, executable='/bin/bash').stdout.strip()
    v = f'{maj}-{mi}'
    !wget -q https://developer.download.nvidia.com/compute/cuda/repos/{ubuntu}/x86_64/cuda-keyring_1.1-1_all.deb && dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null
    !apt-get -qq update > /dev/null && apt-get -qq install -y cuda-minimal-build-{v} cuda-cudart-dev-{v} cuda-cccl-{v} cuda-nvrtc-dev-{v} libcublas-dev-{v} libcusparse-dev-{v} libcusolver-dev-{v} libcurand-dev-{v} > /dev/null
    os.environ['CUDA_HOME'] = f'/usr/local/cuda-{maj}.{mi}'
    os.environ['PATH'] = os.environ['CUDA_HOME'] + '/bin:' + os.environ['PATH']
    !nvcc --version | tail -2

%cd /content
!git clone -q https://github.com/thu-ml/SageAttention.git || true
!git -C SageAttention checkout -q d1a57a5  # the commit measured in results/L4_sage.md
!git -C SageAttention log -1 --format='SageAttention %h %cd'
%cd /content/SageAttention
# torch >= 2.10 headers use C++20 `requires` clauses; SageAttention's setup.py hardcodes -std=c++17
!sed -i 's/-std=c++17/-std=c++20/g' setup.py && grep -n 'std=c++' setup.py && g++ --version | head -1
!TORCH_CUDA_ARCH_LIST=8.9 EXT_PARALLEL=4 MAX_JOBS=8 pip install --no-build-isolation -v . > /content/sage_build.log 2>&1; tail -40 /content/sage_build.log
%cd /content/video-dit-l4
r = subprocess.run([sys.executable, '-c', 'import importlib.metadata as m, sageattention; print("sageattention", m.version("sageattention"))'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
if r.returncode != 0:
    raise RuntimeError('SageAttention did not build or import; stopping here. Send the output above.')

## Runs (16 videos each)

50 steps, the base (~42 min).

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --seeds 0 1 2 3 --out results/q_steps50

20 steps (~19 min), then cache 0.20, the positive control (~12 min).

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --seeds 0 1 2 3 --steps 20 --out results/q_steps20
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --seeds 0 1 2 3 --cache-threshold 0.20 --out results/q_fbc020

Cache 0.10 (~18 min), then 10 steps (~12 min).

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --seeds 0 1 2 3 --cache-threshold 0.10 --out results/q_fbc010
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --seeds 0 1 2 3 --steps 10 --out results/q_steps10

## Q1: seed 0 against the earlier runs

In [ ]:
import json, os
runs = ['q_steps50', 'q_steps20', 'q_fbc020', 'q_fbc010', 'q_steps10']
for r in runs:
    if os.path.isdir(f'results/{r}'):
        !python bench/hash_frames.py results/{r} > /dev/null
for new, old in [('q_steps50', 'pv16_33f'), ('q_steps20', 'steps20_33f'), ('q_fbc010', 'fbc010_33f'), ('q_fbc020', 'fbc020_33f')]:
    if not os.path.exists(f'results/{new}/frames_sha256.json'):
        print(new, 'missing'); continue
    a = json.load(open(f'results/{new}/frames_sha256.json'))
    b = json.load(open(f'results/{old}/frames_sha256.json'))
    same = [p for p in range(4) if a[f'frames_{p}_s0.npy']['sha256'] == b[f'frames_{p}.npy']['sha256']]
    print(f'{new} seed 0 vs {old}: {len(same)} of 4 identical')

## Metrics (~5 min) and the paired table

In [ ]:
done = ' '.join(f'results/{r}' for r in runs if os.path.isdir(f'results/{r}'))
!python bench/quality_metrics.py {done}
tests = ' '.join(f'results/{r}' for r in runs[1:] if os.path.isdir(f'results/{r}'))
!python tools/quality_table.py --base results/q_steps50 {tests} --out results/quality_table.json

In [ ]:
# zip without .npy; the mp4s are kept for the blind side-by-side, judged locally
!zip -qr quality_results.zip {done} results/quality_table.json -x '*.npy' && ls -lh quality_results.zip
from google.colab import files; files.download('quality_results.zip')